In [1]:
#1. Retrieval: gives results from vector store based on query
# def retrieve: Retrieve relevant documents from the vector store in form of a list of dictionaries 
# query is tranformed into an embedding and then used to search the vector store for similar documents

import sys
sys.path.insert(0, '../')

from typing import List, Dict, Any
from src.embedding_manager import EmbeddingManager
from src.vector_store import FaissVectorStore

class RAGRetriever:
    def __init__(self, vector_store: FaissVectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        
        try:
            # Generate embedding for the query
            query_embeddings = self.embedding_manager.generate_embeddings([query])
            query_embedding = query_embeddings[0]
            
            # Search the vector store - returns list of tuples (metadata, distance)
            results = self.vector_store.search(query_embedding, top_k=top_k)
            retrieved_docs = []
            
            for rank, (metadata, distance) in enumerate(results, 1):
                similarity_score = distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': rank,
                        'content': metadata.get('content', ''),
                        'source': metadata.get('source', ''),
                        'similarity_score': float(similarity_score),
                        'metadata': metadata,
                        'rank': rank
                    })
            
            if retrieved_docs:
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}")
            else:
                print(f"No documents retrieved above the score threshold of {score_threshold}")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            import traceback
            traceback.print_exc()
            return []

/Users/merlindconstanzepohl/Desktop/RAG Implementierung/Code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#2. Initialize vector store and embedding manager (load from disk)

try:
    embedding_manager = EmbeddingManager()
    print("EmbeddingManager initialized successfully")
except Exception as e:
    print(f"Failed to initialize EmbeddingManager: {e}")
    embedding_manager = None

try:
    vector_store = FaissVectorStore(embedding_dim=embedding_manager.get_embedding_dimension())
    print(f"Vector store loaded successfully with {len(vector_store.id_to_metadata)} chunks")
except Exception as e:
    print(f"Failed to initialize Vector Store: {e}")
    vector_store = None

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 59070.45it/s]


Loaded embedding model: BAAI/bge-m3. Embedding dimension: 1024
EmbeddingManager initialized successfully
Loaded index with 251 chunks
Vector store loaded successfully with 251 chunks


In [3]:
# 3. Initialize RAGRetriever with instances from ingestion.ipynb

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"
results = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

for doc in results:
    
    print(f"\nRank {doc['rank']}: {doc['similarity_score']:.4f}")
    print(f"Source: {doc['source']}")
    print(f"Content preview: {doc['content'][:200]}...")


Retrieving documents for query: 'What is the economic contribution of the Great Barrier Reef?' with top_k=5 and score_threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]

Retrieved 5 documents above the score threshold of 0.0

Rank 1: 0.5855
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: 44 7. TOTAL ECONOMIC VALUE OF CATCHMENT AREA The total economic contributions of tourism, commercial fishing and cultural & recreational activity to the GBRCA are the sums of the direct contributions ...

Rank 2: 0.5803
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: They are also, at present at least, largely masked by broader economic drivers such as real income growth, the value of the Australian dollar, oil prices and associated effects on airfares, geopolitic...

Rank 3: 0.5795
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: Access Economics notes that estimates of the economic cost of damage to the have already been made. One of these puts th

In [4]:
#4. LLM integration with Ollama

from langchain_ollama import OllamaLLM
from dotenv import load_dotenv
load_dotenv()

#temperature defines how "creative" the model's answers will be --> low value for solid and factual answers
#top_p defines the diversity of the output ("creativity") --> low value for more deterministic answers (better not 0.0 --> often ignored and set to default)
llm = OllamaLLM(model="llama3", temperature=0.1, top_p=0.1)


In [5]:
#5. RAG function for information retrieval with minimal instructions


def retrieval_query(query: str, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 5, score_threshold: float = 0.3, return_context=False):

    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=score_threshold)
    
    if not results:
        return "No relevant documents found to answer the question."

    confidence = max(doc['similarity_score'] for doc in results)

    sources = []
    for doc in results:
        clean_preview = doc['content'][:150].replace('\n', ' ').strip()
        sources.append({
            'source': doc['source'],
            'score': round(float(doc['similarity_score']), 3),
            'preview': f"{clean_preview}..."
        })
    
    
    # in case of low confidence,choose to return the sources with a note about low confidence instead of an answer to avoid errors
    if confidence < score_threshold:
        return {
            'answer': "The found documents are not sufficiently relevant. I refuse to answer to avoid errors.",
            'sources': sources,
            'confidence': round(float(confidence), 3)
        }
    
    context = "\n\n".join([doc['content'] for doc in results])
    
    # generate answer    
    prompt = f"""
    Use the following context to answer the question concisely and factually. 
    Do not say where the information comes from, just give the answer. 
    If the provided texts mention different numbers or information for the same topic, list them separately. 

        Context: {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke(prompt)

    output = {
        'response': response,
        'sources': sources,
        'confidence': round(float(confidence), 3)
    }
    
    if return_context:
        output['context'] = context
    
    return output
 

In [6]:
#6. Output query with sources and confidence score

result = retrieval_query("How much will the Earth still warm up?", rag_retriever, llm, return_context=True)

print("\n" + "_"*100)
print("")
print("Answer:")
print("")
print(result['response'])

print("\n" + "_"*100)
print("")
print("Sources:")
print("")
for i, source in enumerate(result['sources'], 1):
    print(f"\n[Source {i}]")
    print(f"File: {source['source']}")
    print(f"Score: {source['score']}")
    print(f"Preview:{source['preview']}")

print("\n" + "_"*100)
print(f"Confidence Score: {result['confidence']}")
print("_"*100)

Retrieving documents for query: 'How much will the Earth still warm up?' with top_k=5 and score_threshold=0.3


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.58it/s]


Retrieved 5 documents above the score threshold of 0.3

____________________________________________________________________________________________________

Answer:

According to the provided context:

* Since the start of the 20th century, the global average surface temperature has risen between 0.6 and 0.7 degrees Celsius.
* Since then, the global average surface temperature has increased by 0.18 degrees Celsius per decade.
* In the 1980s, the increase in average global surface temperatures was even faster.

As for future warming, there is no specific prediction mentioned in the context. However, it is noted that some projections suggest the Arctic Ocean could be ice-free or nearly so by the end of the 21st century.

____________________________________________________________________________________________________

Sources:


[Source 1]
File: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Score: 0.629
Preview:As part of this focus